## Setup environments in Colab


In [1]:
import torch

In [2]:
torch.cuda.get_device_name(0)

'Tesla T4'

In [3]:
!rm -rf jihun-personal-gym/
!git clone https://github.com/jihunchoi/jihun-personal-gym

Cloning into 'jihun-personal-gym'...
remote: Enumerating objects: 145, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 145 (delta 55), reused 140 (delta 50), pack-reused 0 (from 0)
Receiving objects: 100% (145/145), 148.98 KiB | 681.00 KiB/s, done.
Resolving deltas: 100% (55/55), done.


In [4]:
%cd /content/jihun-personal-gym/diffusion-models/ddpm

/content/jihun-personal-gym/diffusion-models/ddpm


In [5]:
!uv pip install --system .

Using Python 3.12.12 environment at: /usr
Resolved 123 packages in 838ms                                       
Prepared 9 packages in 1.30s                                             
Installed 9 packages in 79ms                                
 + async-lru==2.2.0
 + ddpm==0.1.0 (from file:///content/jihun-personal-gym/diffusion-models/ddpm)
 + diffusion-common==0.1.0 (from file:///content/jihun-personal-gym/diffusion-models/diffusion-common)
 + jedi==0.19.2
 + json5==0.13.0
 + jupyter-lsp==2.3.0
 + jupyterlab==4.5.5
 + jupyterlab-server==2.28.0
 + pyrefly==0.55.0


In [6]:
from google.colab import drive

drive.mount("/content/drive/")

Mounted at /content/drive/


In [12]:
!ls /content/drive/MyDrive/Workspace/jihun-personal-gym/diffusion-models/ddpm/scripts

train_cifar10.py


## Data


In [13]:
from torchvision import transforms
from torchvision.datasets import CIFAR10

transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ]
)

dataset = CIFAR10(root="/content/data", train=True, download=True, transform=transform)

100%|██████████| 170M/170M [02:38<00:00, 1.08MB/s]


In [14]:
from torch.utils.data import DataLoader

dataloader = DataLoader(dataset, batch_size=512, shuffle=True, num_workers=2)

## Hyperparameters


In [15]:
# ResUNet
input_dim = 3  # RGB
output_dim = 3
layer_dims = [64, 128, 256]
bridge_dim = 512
time_embedding_dim = 128

# DDPM
num_steps = 1000  # Number of steps
# betas: Typically 0.0001 to 0.02, but our parameterization defines
# each beta as the standard deviation of the noise, instead of the variance.
betas = torch.linspace(0.01, 0.15, num_steps).tolist()

# Training
num_epochs = 20
checkpoint_dir = "/content/drive/MyDrive/Workspace/jihun-personal-gym/diffusion-models/ddpm/scripts/checkpoints/cifar10"

In [ ]:
import os

import yaml

hparams = {
    "noise_prediction_model": {
        "input_dim": input_dim,
        "output_dim": output_dim,
        "layer_dims": layer_dims,
        "bridge_dim": bridge_dim,
        "time_embedding_dim": time_embedding_dim,
    },
    "ddpm": {
        "num_steps": num_steps,
        "betas": betas,
    },
    "training": {
        "num_epochs": num_epochs,
    },
}

with open(os.path.join(checkpoint_dir, "hparams.yaml"), "w") as f:
    yaml.dump(hparams, f)

## Model


In [16]:
from diffusion_common.decoders import ResUNet

from ddpm.forward import DDPMForwardProcess

noise_prediction_model = ResUNet(
    input_dim=input_dim,
    output_dim=output_dim,
    layer_dims=layer_dims,
    bridge_dim=bridge_dim,
    time_embedding_dim=time_embedding_dim,
)

forward_process = DDPMForwardProcess(betas=betas)

## Trainer & optimizer


In [17]:
from torch import optim

from ddpm.trainer import DDPMTrainer

trainer = DDPMTrainer(
    noise_prediction_model=noise_prediction_model, forward_process=forward_process
)
optimizer = optim.Adam(noise_prediction_model.parameters(), lr=1e-4)

## Train


In [18]:
trainer.train(
    dataloader=dataloader,
    optimizer=optimizer,
    num_epochs=num_epochs,
    checkpoint_dir=checkpoint_dir,
    checkpoint_freq=1,
    device="cuda",
)

Epoch 1/20: 100%|██████████| 98/98 [01:00<00:00,  1.62it/s, loss=0.1287]


Checkpoint saved to /content/drive/MyDrive/Workspace/jihun-personal-gym/diffusion-models/ddpm/scripts/checkpoints/cifar10/checkpoint_epoch_1.pt


Epoch 2/20: 100%|██████████| 98/98 [01:06<00:00,  1.48it/s, loss=0.1010]


Checkpoint saved to /content/drive/MyDrive/Workspace/jihun-personal-gym/diffusion-models/ddpm/scripts/checkpoints/cifar10/checkpoint_epoch_2.pt


Epoch 3/20: 100%|██████████| 98/98 [01:08<00:00,  1.42it/s, loss=0.0902]


...

Checkpoint saved to /content/drive/MyDrive/Workspace/jihun-personal-gym/diffusion-models/ddpm/scripts/checkpoints/cifar10/checkpoint_epoch_18.pt


Epoch 19/20: 100%|██████████| 98/98 [01:09<00:00,  1.41it/s, loss=0.0685]


Checkpoint saved to /content/drive/MyDrive/Workspace/jihun-personal-gym/diffusion-models/ddpm/scripts/checkpoints/cifar10/checkpoint_epoch_19.pt


Epoch 20/20: 100%|██████████| 98/98 [01:09<00:00,  1.42it/s, loss=0.0623]


Checkpoint saved to /content/drive/MyDrive/Workspace/jihun-personal-gym/diffusion-models/ddpm/scripts/checkpoints/cifar10/checkpoint_epoch_20.pt
